# Bradford Bulls — Standardized Frame Extraction Pipeline
## Production v1.0 — ALL team members use this notebook

### RULES
1. **DO NOT** modify any cell except **Cell 1 (Your Config)**
2. **DO NOT** write your own extraction script
3. **DO NOT** manually delete or filter frames after extraction
4. If you encounter issues, report to lead — do not improvise

### Video Naming Convention (on Google Drive)
Upload your video to `Bradford_Bulls/videos/` with this exact format:
```
M{number}_{kit_color}_{resolution}.mp4
```

| Example | Meaning |
|---------|---------|
| `M01_white_1080p.mp4` | Match 1, home white kit, 1080p |
| `M02_black_1080p.mp4` | Match 2, away black kit, 1080p |
| `M03_white_720p.mp4`  | Match 3, home white kit, 720p |
| `M06_white_1080p_highlight.mp4` | Match 6, highlight reel |

### Pipeline
```
Video (Google Drive)
  → L1: Temporal sampling (1 FPS)
  → L2: Scene change detection (pHash + SSIM)
  → L3: Player presence + size filter (YOLO)
  → L4: Quality scoring + diversity selection
  → Save to Google Drive (frames + metadata CSV)
  → Ready for Roboflow upload
```

### Output Structure
```
Bradford_Bulls/
├── videos/
│   └── M01_white_1080p.mp4
├── frames/
│   └── M01_white_1080p/
│       ├── M01_white_1080p_000045_00m45s.jpg
│       ├── M01_white_1080p_000132_02m12s.jpg
│       └── ...
├── metadata/
│   └── M01_white_1080p_index.csv
└── reports/
    └── M01_white_1080p_report.txt
```

## 0. Install Dependencies
Run this cell first, then **Restart Runtime** before continuing.

In [ ]:
!pip install -q ultralytics>=8.3.0 imagehash scikit-image pillow opencv-python-headless roboflow
print("\nDone. Please RESTART RUNTIME now (Runtime > Restart runtime), then continue from Cell 1.")

## 1. YOUR CONFIG (ONLY cell you edit)

**Instructions:**
1. Set `MEMBER_NAME` to your name (no spaces, lowercase)
2. Set `VIDEO_FILENAME` to **exactly** the filename of your video on Google Drive
3. Set `TEST_MODE = True` for first run to verify everything works, then set `False` for full run

In [ ]:
# ============================================================
# YOUR CONFIG — Edit ONLY these 3 lines
# ============================================================

MEMBER_NAME = "your_name"                    # e.g. "hoa", "minh", "tuan"
VIDEO_FILENAME = "M01_white_1080p.mp4"       # Exact filename on Google Drive
TEST_MODE = True                             # True = test with 2000 frames, False = full video

# ============================================================
# DO NOT EDIT BELOW THIS LINE
# ============================================================

## 2. Setup Environment & Mount Google Drive

In [ ]:
import torch
import cv2
import os
import json
import csv
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import imagehash
from pathlib import Path
from collections import Counter
from datetime import datetime
from PIL import Image
from tqdm import tqdm
from ultralytics import YOLO
from skimage.metrics import structural_similarity as compare_ssim

# --- GPU Check ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("WARNING: No GPU detected. This will be slow.")

# --- Mount Google Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- Directory Structure ---
DRIVE_ROOT = Path("/content/drive/MyDrive/Bradford_Bulls")
VIDEOS_DIR = DRIVE_ROOT / "videos"
FRAMES_DIR = DRIVE_ROOT / "frames"
METADATA_DIR = DRIVE_ROOT / "metadata"
REPORTS_DIR = DRIVE_ROOT / "reports"

for d in [VIDEOS_DIR, FRAMES_DIR, METADATA_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Derive match_id from video filename ---
# Expected format: M01_white_1080p.mp4 or M01_white_1080p_highlight.mp4
MATCH_ID = Path(VIDEO_FILENAME).stem  # e.g. "M01_white_1080p"

# Output directories for THIS video
FRAMES_OUTPUT_DIR = FRAMES_DIR / MATCH_ID
FRAMES_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Validate video exists ---
VIDEO_PATH = VIDEOS_DIR / VIDEO_FILENAME
if not VIDEO_PATH.exists():
    available = list(VIDEOS_DIR.glob("*.mp4"))
    print(f"ERROR: '{VIDEO_FILENAME}' not found in {VIDEOS_DIR}")
    print(f"\nAvailable videos:")
    for v in available:
        size_mb = v.stat().st_size / 1e6
        print(f"  - {v.name} ({size_mb:.0f} MB)")
    if not available:
        print("  (none — please upload your video first)")
    raise FileNotFoundError(f"Video not found: {VIDEO_PATH}")

print(f"\nVideo found: {VIDEO_PATH.name} ({VIDEO_PATH.stat().st_size/1e6:.0f} MB)")
print(f"Member: {MEMBER_NAME}")
print(f"Match ID: {MATCH_ID}")
print(f"Output: {FRAMES_OUTPUT_DIR}")

## 3. Load Video & Display Info

In [ ]:
# --- Read video metadata ---
cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise ValueError(f"Cannot open video: {VIDEO_PATH}")

FPS = cap.get(cv2.CAP_PROP_FPS)
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
DURATION_SEC = TOTAL_FRAMES / FPS
cap.release()

# --- Standardized extraction parameters (LOCKED) ---
TARGET_FPS = 1                      # 1 frame per second
PHASH_THRESHOLD = 10                # Perceptual hash dedup threshold
SSIM_THRESHOLD = 0.92               # Scene similarity threshold
PERSON_CONFIDENCE = 0.5             # YOLO person detection confidence
MIN_PERSONS = 1                     # Minimum persons in frame
MIN_MAX_PERSON_AREA_RATIO = 0.03    # Reject wide shots (person < 3% of frame)
MIN_SHARPNESS = 0.25                # Sharpness threshold (0-1)
ENABLE_PITCH_GREEN_FILTER = True    # Filter non-pitch frames
PITCH_ROI_Y_START = 0.40            # Analyze bottom 60% of frame
MIN_PITCH_GREEN_RATIO = 0.06        # Min grass-green pixel ratio
DEDUP_HASH_THRESH = 6               # Hash distance for final dedup
TIME_BUCKET_SEC = 10                # Diversity: max frames per N-second window
PERSON_DETECTION_MODEL = "yolo26m.pt"  # Model for person detection (medium for accuracy)

JPEG_QUALITY = 95                   # Output quality
INCLUDE_MOTION_BLUR_FRAMES = True   # Keep some blurry frames for training diversity
MOTION_BLUR_SHARPNESS_MIN = 0.12    # Floor for motion blur frames (below this = unusable)
MOTION_BLUR_QUOTA = 0.15            # Max 15% of selected frames can be motion-blurred

# Frame interval based on video FPS
FRAME_INTERVAL = max(1, int(FPS / TARGET_FPS))

# Test mode limits
if TEST_MODE:
    MAX_SCAN_FRAMES = 2000
    TARGET_SELECTED = 30
    print(f"TEST MODE: scanning {MAX_SCAN_FRAMES} frames, selecting {TARGET_SELECTED}")
else:
    MAX_SCAN_FRAMES = TOTAL_FRAMES
    TARGET_SELECTED = None  # Select all that pass filters
    print(f"PRODUCTION MODE: scanning all {TOTAL_FRAMES:,} frames")

print(f"\n{'='*60}")
print(f"  VIDEO INFO")
print(f"{'='*60}")
print(f"  File:        {VIDEO_PATH.name}")
print(f"  Duration:    {DURATION_SEC/60:.1f} min ({DURATION_SEC:.0f} sec)")
print(f"  Resolution:  {W}x{H}")
print(f"  FPS:         {FPS:.0f}")
print(f"  Total:       {TOTAL_FRAMES:,} frames")
print(f"  Sample rate: 1 frame every {FRAME_INTERVAL} frames ({TARGET_FPS} FPS)")
print(f"  Estimated L1 frames: ~{TOTAL_FRAMES // FRAME_INTERVAL:,}")
print(f"{'='*60}")

## 4. Helper Functions (Locked)

In [ ]:
def fmt_timestamp(sec):
    """Convert seconds to HH:MM:SS string for filenames."""
    h = int(sec // 3600)
    m = int((sec % 3600) // 60)
    s = int(sec % 60)
    return f"{h:02d}h{m:02d}m{s:02d}s" if h > 0 else f"{m:02d}m{s:02d}s"


def compute_sharpness(frame):
    """Multi-metric sharpness score (0-1). Higher = sharper."""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    # Laplacian variance
    lap = cv2.Laplacian(gray, cv2.CV_64F).var()
    # Tenengrad (Sobel gradient magnitude)
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    ten = np.mean(gx**2 + gy**2)
    # Normalized variance
    mean_val = gray.mean()
    nvar = gray.var() / (mean_val + 1e-6)
    # Weighted combination, capped at 1.0
    score = (
        min(lap / 300, 1.0) * 0.40 +
        min(ten / 5000, 1.0) * 0.35 +
        min(nvar / 50, 1.0) * 0.25
    )
    return round(score, 4)


def compute_player_sharpness(frame, person_detections):
    """Compute sharpness ONLY within person bounding boxes.
    This avoids false high scores from sharp text overlays (scoreboard, graphics).
    Returns the average sharpness across all detected persons.
    """
    if not person_detections:
        return compute_sharpness(frame)

    h, w = frame.shape[:2]
    sharpness_scores = []

    for det in person_detections:
        x1, y1, x2, y2 = det["bbox"]
        # Clamp to frame bounds
        x1i, y1i = int(max(0, x1)), int(max(0, y1))
        x2i, y2i = int(min(w, x2)), int(min(h, y2))
        if x2i <= x1i or y2i <= y1i:
            continue

        crop = frame[y1i:y2i, x1i:x2i]
        if crop.size == 0:
            continue

        sharpness_scores.append(compute_sharpness(crop))

    if not sharpness_scores:
        return compute_sharpness(frame)

    # Use the MAX player sharpness (best visible player)
    return round(max(sharpness_scores), 4)


def compute_phash(frame_bgr):
    """Compute perceptual hash of a frame."""
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    return imagehash.phash(Image.fromarray(frame_rgb))


def compute_ssim(frame1, frame2):
    """Compute structural similarity between two frames."""
    gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)
    if gray1.shape != gray2.shape:
        gray2 = cv2.resize(gray2, (gray1.shape[1], gray1.shape[0]))
    return float(compare_ssim(gray1, gray2))


def compute_pitch_green_ratio(frame_bgr, roi_y_start=0.40):
    """Fraction of grass-green pixels in bottom ROI (0-1)."""
    h, w = frame_bgr.shape[:2]
    y0 = int(max(0, min(h - 1, roi_y_start * h)))
    roi = frame_bgr[y0:h, 0:w]
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    lower = np.array([35, 40, 40])
    upper = np.array([95, 255, 255])
    mask = cv2.inRange(hsv, lower, upper)
    return float(mask.mean() / 255.0)


def detect_persons_batch(model, frames, device, confidence=0.5):
    """Batch detect persons, return list of detection dicts per frame."""
    results = model.predict(
        frames, classes=[0], conf=confidence,
        device=device, verbose=False, batch=len(frames)
    )
    all_detections = []
    for res in results:
        detections = []
        if res.boxes is not None and len(res.boxes) > 0:
            xyxy = res.boxes.xyxy.cpu().numpy()
            confs = res.boxes.conf.cpu().numpy()
            for i in range(len(xyxy)):
                x1, y1, x2, y2 = xyxy[i]
                detections.append({
                    "bbox": [float(x1), float(y1), float(x2), float(y2)],
                    "conf": float(confs[i]),
                    "area": float((x2 - x1) * (y2 - y1)),
                })
        all_detections.append(detections)
    return all_detections


print("Helper functions loaded.")

## 5. Content-Aware Frame Extraction (2-Pass)

### Why not fixed-rate sampling (1 FPS)?
Fixed-rate sampling is blind to content. A video with 80% wide shots gives 80% useless frames.

### New approach: Find quality moments, then pick the best frame

**Pass 1 — Fast scan (every 5th frame):**
Scan entire video quickly. For each sampled frame, compute only `max_person_area_ratio` (how big is the largest person). This creates a "zoom timeline" — we know WHEN the camera is zoomed into players.

**Segment detection:**
Find contiguous segments where `max_person_area_ratio > threshold`. These are close-ups, replays, celebrations — moments where logos are actually visible.

**Pass 2 — Best frame per segment:**
Within each quality segment, read ALL frames and pick the **sharpest** one (measured on person bbox). One segment may be 2-5 seconds long, but we only keep the single best frame.

```
Video timeline:
  [wide][wide][CLOSE-UP segment][wide][REPLAY segment][wide][CELEBRATION]
                    ↓                       ↓                    ↓
              best frame #1          best frame #2         best frame #3

Result: fewer frames, but MUCH higher quality.
```

In [ ]:
# ============================================================
# PASS 1: Fast scan — build "zoom timeline"
# ============================================================
print("Loading YOLO model...")
yolo_model = YOLO(PERSON_DETECTION_MODEL)
print(f"Model loaded on {DEVICE}")

cap = cv2.VideoCapture(str(VIDEO_PATH))
frame_area = W * H

# Scan every SCAN_INTERVAL-th frame for speed
SCAN_INTERVAL = 5  # check every 5th frame (~6 FPS for 30fps video)

zoom_timeline = []  # (frame_num, max_person_area_ratio, n_persons)
scan_batch_frames = []
scan_batch_nums = []
BATCH_SIZE = 32

scan_limit = MAX_SCAN_FRAMES if TEST_MODE else TOTAL_FRAMES
pbar = tqdm(total=scan_limit, desc="Pass 1: Scanning zoom levels", unit="fr")

frame_num = 0
while True:
    ret, frame = cap.read()
    if not ret or frame_num >= scan_limit:
        break
    pbar.update(1)

    if frame_num % SCAN_INTERVAL != 0:
        frame_num += 1
        continue

    scan_batch_frames.append(frame)
    scan_batch_nums.append(frame_num)

    if len(scan_batch_frames) >= BATCH_SIZE:
        results = yolo_model.predict(
            scan_batch_frames, classes=[0], conf=PERSON_CONFIDENCE,
            device=DEVICE, verbose=False
        )
        for fn, res in zip(scan_batch_nums, results):
            n_persons = len(res.boxes) if res.boxes is not None else 0
            max_ratio = 0.0
            if n_persons > 0:
                areas = []
                for box in res.boxes:
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                    areas.append((x2-x1)*(y2-y1) / frame_area)
                max_ratio = max(areas)
            zoom_timeline.append((fn, max_ratio, n_persons))
        scan_batch_frames, scan_batch_nums = [], []

    frame_num += 1

# Process remaining
if scan_batch_frames:
    results = yolo_model.predict(
        scan_batch_frames, classes=[0], conf=PERSON_CONFIDENCE,
        device=DEVICE, verbose=False
    )
    for fn, res in zip(scan_batch_nums, results):
        n_persons = len(res.boxes) if res.boxes is not None else 0
        max_ratio = 0.0
        if n_persons > 0:
            areas = []
            for box in res.boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                areas.append((x2-x1)*(y2-y1) / frame_area)
            max_ratio = max(areas)
        zoom_timeline.append((fn, max_ratio, n_persons))

pbar.close()
cap.release()

# ============================================================
# SEGMENT DETECTION — find quality windows
# ============================================================
# A "quality segment" = contiguous frames where person is close enough
MIN_SEGMENT_FRAMES = 2   # at least 2 scan points (~0.3s)
SEGMENT_GAP_TOLERANCE = 3 # allow gaps of up to 3 scan points

segments = []
current_seg = []

for i, (fn, ratio, n_p) in enumerate(zoom_timeline):
    if ratio >= MIN_MAX_PERSON_AREA_RATIO and n_p >= MIN_PERSONS:
        current_seg.append((fn, ratio, n_p))
    else:
        # Check if this is a small gap within a segment
        if current_seg and i + 1 < len(zoom_timeline):
            # Look ahead
            next_fn, next_ratio, next_np = zoom_timeline[min(i+1, len(zoom_timeline)-1)]
            if next_ratio >= MIN_MAX_PERSON_AREA_RATIO and len(current_seg) >= MIN_SEGMENT_FRAMES:
                continue  # tolerate this gap
        if len(current_seg) >= MIN_SEGMENT_FRAMES:
            segments.append(current_seg)
        current_seg = []

if len(current_seg) >= MIN_SEGMENT_FRAMES:
    segments.append(current_seg)

# Compute segment stats
total_segment_frames = sum(
    seg[-1][0] - seg[0][0] + SCAN_INTERVAL for seg in segments
)

print(f"\n{'='*60}")
print(f"  PASS 1 RESULTS — {MATCH_ID}")
print(f"{'='*60}")
print(f"  Frames scanned:     {len(zoom_timeline):,} (every {SCAN_INTERVAL}th frame)")
print(f"  Quality segments:   {len(segments)}")
print(f"  Segment coverage:   {total_segment_frames/max(scan_limit,1)*100:.1f}% of video")
if segments:
    avg_len = np.mean([(s[-1][0]-s[0][0])/FPS for s in segments])
    print(f"  Avg segment length: {avg_len:.1f}s")
    max_ratio = max(max(r for _, r, _ in s) for s in segments)
    print(f"  Best zoom ratio:    {max_ratio:.3f}")
print(f"{'='*60}")

if not segments:
    print("\nWARNING: No quality segments found!")
    print("Try lowering MIN_MAX_PERSON_AREA_RATIO in config.")

## 6. Pass 2 — Extract Best Frame per Segment

For each quality segment found in Pass 1:
1. Read ALL frames in that segment (not just 1 FPS)
2. Run person detection on each
3. Measure sharpness on person bounding boxes
4. Pick the **single best frame** (highest quality score)
5. Also pick a **second-best** if segment is long (>3 seconds)

This guarantees we always get the sharpest possible frame from every close-up moment.

In [ ]:
# ============================================================
# PASS 2: Extract best frame from each quality segment
# ============================================================
cap = cv2.VideoCapture(str(VIDEO_PATH))

candidates = []
stats = {
    "segments_processed": 0,
    "frames_analyzed_pass2": 0,
    "frames_skipped_pitch": 0,
    "frames_skipped_blurry": 0,
}

# How many frames to pick per segment
FRAMES_PER_SHORT_SEGMENT = 1   # segments < 3 seconds
FRAMES_PER_LONG_SEGMENT = 2    # segments >= 3 seconds
LONG_SEGMENT_THRESHOLD = 3.0   # seconds

for seg_idx, segment in enumerate(tqdm(segments, desc="Pass 2: Extracting best frames")):
    seg_start_frame = segment[0][0]
    seg_end_frame = segment[-1][0] + SCAN_INTERVAL
    seg_duration = (seg_end_frame - seg_start_frame) / FPS

    # Determine how many frames to pick from this segment
    n_pick = FRAMES_PER_LONG_SEGMENT if seg_duration >= LONG_SEGMENT_THRESHOLD else FRAMES_PER_SHORT_SEGMENT

    # Read ALL frames in this segment
    cap.set(cv2.CAP_PROP_POS_FRAMES, seg_start_frame)
    segment_candidates = []

    for fn in range(seg_start_frame, min(seg_end_frame, TOTAL_FRAMES)):
        ret, frame = cap.read()
        if not ret:
            break
        stats["frames_analyzed_pass2"] += 1

        # Pitch filter
        pitch_green = compute_pitch_green_ratio(frame, PITCH_ROI_Y_START)
        if ENABLE_PITCH_GREEN_FILTER and pitch_green < MIN_PITCH_GREEN_RATIO:
            stats["frames_skipped_pitch"] += 1
            continue

        # Person detection on this specific frame
        results = yolo_model.predict(
            frame, classes=[0], conf=PERSON_CONFIDENCE,
            device=DEVICE, verbose=False
        )

        detections = []
        if results and results[0].boxes is not None:
            for box in results[0].boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                detections.append({
                    "bbox": [float(x1), float(y1), float(x2), float(y2)],
                    "conf": float(box.conf[0].cpu().numpy()),
                    "area": float((x2-x1)*(y2-y1)),
                })

        if len(detections) < MIN_PERSONS:
            continue

        max_person_area_ratio = max(d["area"] / frame_area for d in detections)
        if max_person_area_ratio < MIN_MAX_PERSON_AREA_RATIO:
            continue

        # Sharpness on person regions
        sharpness = compute_player_sharpness(frame, detections)

        if sharpness < MOTION_BLUR_SHARPNESS_MIN:
            stats["frames_skipped_blurry"] += 1
            continue

        total_player_area = sum(d["area"] for d in detections)
        player_coverage = min(total_player_area / frame_area, 1.0)
        is_motion_blur = sharpness < MIN_SHARPNESS

        # Quality score: sharpness is king for frame selection
        score = (
            sharpness * 0.50 +
            max_person_area_ratio * 0.30 +
            min(len(detections) / 5.0, 1.0) * 0.20
        )

        timestamp_sec = fn / FPS
        segment_candidates.append({
            "frame_num": fn,
            "timestamp_sec": round(timestamp_sec, 2),
            "timestamp_hms": fmt_timestamp(timestamp_sec),
            "n_persons": len(detections),
            "sharpness": sharpness,
            "is_motion_blur": is_motion_blur,
            "player_coverage": round(player_coverage, 4),
            "max_person_area_ratio": round(max_person_area_ratio, 4),
            "pitch_green_ratio": round(pitch_green, 4),
            "score": round(score, 4),
            "segment_idx": seg_idx,
            "segment_duration": round(seg_duration, 1),
        })

    stats["segments_processed"] += 1

    # Pick top N from this segment
    if segment_candidates:
        segment_candidates.sort(key=lambda x: x["score"], reverse=True)

        # Pick best frame
        candidates.append(segment_candidates[0])

        # For long segments, pick second best (but at least 1s apart from first)
        if n_pick >= 2 and len(segment_candidates) > 1:
            best_ts = segment_candidates[0]["timestamp_sec"]
            for sc in segment_candidates[1:]:
                if abs(sc["timestamp_sec"] - best_ts) >= 1.0:
                    candidates.append(sc)
                    break

cap.release()

# Sort by timestamp
candidates.sort(key=lambda x: x["timestamp_sec"])

# Motion blur stats
blur_count = sum(1 for c in candidates if c["is_motion_blur"])

print(f"\n{'='*60}")
print(f"  PASS 2 RESULTS — {MATCH_ID}")
print(f"{'='*60}")
print(f"  Segments processed:    {stats['segments_processed']}")
print(f"  Frames analyzed (P2):  {stats['frames_analyzed_pass2']:,}")
print(f"  Skipped (no pitch):    {stats['frames_skipped_pitch']:,}")
print(f"  Skipped (too blurry):  {stats['frames_skipped_blurry']:,}")
print(f"  FINAL FRAMES:          {len(candidates)}")
print(f"    Sharp:               {len(candidates) - blur_count}")
print(f"    Motion blur:         {blur_count}")
if candidates:
    scores = [c['score'] for c in candidates]
    sharpness_vals = [c['sharpness'] for c in candidates]
    print(f"  Score range:           {min(scores):.3f} — {max(scores):.3f}")
    print(f"  Sharpness range:       {min(sharpness_vals):.3f} — {max(sharpness_vals):.3f}")
    print(f"  Time span:             {candidates[0]['timestamp_hms']} — {candidates[-1]['timestamp_hms']}")
print(f"{'='*60}")

# In test mode, limit to TARGET_SELECTED
if TEST_MODE and TARGET_SELECTED and len(candidates) > TARGET_SELECTED:
    candidates.sort(key=lambda x: x["score"], reverse=True)
    candidates = candidates[:TARGET_SELECTED]
    candidates.sort(key=lambda x: x["timestamp_sec"])
    print(f"\nTest mode: kept top {TARGET_SELECTED} frames by score")

## 7. Save Frames & Metadata to Google Drive

Saves frames with standardized naming:
```
{MATCH_ID}_{frame_num:06d}_{timestamp}.jpg
```

Also saves a detailed CSV index for review quality.

In [ ]:
# --- Save frames ---
cap = cv2.VideoCapture(str(VIDEO_PATH))
rows = []
saved_count = 0

for idx, meta in enumerate(tqdm(candidates, desc="Saving frames")):
    cap.set(cv2.CAP_PROP_POS_FRAMES, meta["frame_num"])
    ret, frame = cap.read()
    if not ret:
        continue

    fname = f"{MATCH_ID}_{meta['frame_num']:06d}_{meta['timestamp_hms']}.jpg"
    fpath = FRAMES_OUTPUT_DIR / fname
    cv2.imwrite(str(fpath), frame, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
    saved_count += 1

    rows.append({
        "id": idx + 1,
        "filename": fname,
        "frame_num": meta["frame_num"],
        "timestamp_sec": meta["timestamp_sec"],
        "timestamp_hms": meta["timestamp_hms"],
        "n_persons": meta["n_persons"],
        "sharpness": meta["sharpness"],
        "is_motion_blur": meta["is_motion_blur"],
        "player_coverage": meta["player_coverage"],
        "max_person_area_ratio": meta["max_person_area_ratio"],
        "pitch_green_ratio": meta["pitch_green_ratio"],
        "score": meta["score"],
        "segment_idx": meta.get("segment_idx", -1),
        "segment_duration": meta.get("segment_duration", 0),
        "match_id": MATCH_ID,
        "video_file": VIDEO_FILENAME,
        "resolution": f"{W}x{H}",
        "member": MEMBER_NAME,
        "extracted_at": datetime.now().isoformat(),
    })

cap.release()

# --- Save metadata CSV ---
df = pd.DataFrame(rows)
csv_path = METADATA_DIR / f"{MATCH_ID}_index.csv"
df.to_csv(csv_path, index=False)

print(f"\nSaved {saved_count} frames to: {FRAMES_OUTPUT_DIR}")
print(f"Metadata CSV: {csv_path}")

# --- Save extraction report ---
sharp_count = len([r for r in rows if not r.get("is_motion_blur")])
blur_count = len([r for r in rows if r.get("is_motion_blur")])

report_lines = [
    "FRAME EXTRACTION REPORT",
    "=" * 50,
    f"Date:           {datetime.now().strftime('%Y-%m-%d %H:%M')}",
    f"Member:         {MEMBER_NAME}",
    f"Video:          {VIDEO_FILENAME}",
    f"Match ID:       {MATCH_ID}",
    f"Resolution:     {W}x{H}",
    f"Duration:       {DURATION_SEC/60:.1f} min",
    f"FPS:            {FPS:.0f}",
    f"Total frames:   {TOTAL_FRAMES:,}",
    "",
    "PIPELINE STATS (2-Pass Content-Aware)",
    "=" * 50,
    f"Pass 1 scan points:     {len(zoom_timeline):,}",
    f"Quality segments found: {len(segments)}",
    f"Pass 2 frames analyzed: {stats['frames_analyzed_pass2']:,}",
    f"  Skipped (no pitch):   {stats['frames_skipped_pitch']:,}",
    f"  Skipped (too blurry): {stats['frames_skipped_blurry']:,}",
    f"Final saved:            {saved_count}",
    f"  Sharp:                {sharp_count}",
    f"  Motion blur:          {blur_count}",
    "",
    "PARAMETERS (locked)",
    "=" * 50,
    f"PERSON_CONFIDENCE:       {PERSON_CONFIDENCE}",
    f"MIN_PERSONS:             {MIN_PERSONS}",
    f"MIN_MAX_PERSON_AREA:     {MIN_MAX_PERSON_AREA_RATIO}",
    f"MIN_SHARPNESS:           {MIN_SHARPNESS}",
    f"MOTION_BLUR_MIN:         {MOTION_BLUR_SHARPNESS_MIN}",
    f"PITCH_GREEN_FILTER:      {ENABLE_PITCH_GREEN_FILTER}",
    f"MIN_PITCH_GREEN_RATIO:   {MIN_PITCH_GREEN_RATIO}",
    f"PERSON_DETECTION_MODEL:  {PERSON_DETECTION_MODEL}",
]

report_path = REPORTS_DIR / f"{MATCH_ID}_report.txt"
with open(report_path, "w") as f:
    f.write("\n".join(report_lines))

print(f"Report: {report_path}")

## 8. Preview Extracted Frames

Shows a grid of sample frames. Check:
- Are players clearly visible?
- Is there variety in camera angles and timestamps?
- Are motion blur frames reasonable (logo partially visible)?

In [ ]:
# --- Preview: Top scored frames ---
frames_on_disk = sorted(FRAMES_OUTPUT_DIR.glob(f"{MATCH_ID}_*.jpg"))

if not frames_on_disk:
    print("No frames saved yet.")
else:
    # Show top 12 by score and bottom 4 (motion blur examples)
    df_preview = df.sort_values("score", ascending=False)

    # Top 12 best
    top_n = min(12, len(df_preview))
    fig, axes = plt.subplots(3, 4, figsize=(20, 15))
    axes = axes.flatten()

    for i, (_, row) in enumerate(df_preview.head(top_n).iterrows()):
        fpath = FRAMES_OUTPUT_DIR / row["filename"]
        if fpath.exists():
            img = cv2.imread(str(fpath))
            axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            blur_tag = " [BLUR]" if row["is_motion_blur"] else ""
            axes[i].set_title(
                f"{row['timestamp_hms']} | score={row['score']:.2f} | "
                f"sharp={row['sharpness']:.2f} | players={row['n_persons']}{blur_tag}",
                fontsize=8
            )
        axes[i].axis("off")
    for j in range(top_n, len(axes)):
        axes[j].axis("off")

    plt.suptitle(f"TOP {top_n} frames — {MATCH_ID}", fontsize=14)
    plt.tight_layout()
    plt.show()

    # Show motion blur frames if any
    blur_frames = df_preview[df_preview["is_motion_blur"] == True]
    if len(blur_frames) > 0:
        n_blur_show = min(4, len(blur_frames))
        fig, axes = plt.subplots(1, n_blur_show, figsize=(5*n_blur_show, 5))
        if n_blur_show == 1:
            axes = [axes]
        for i, (_, row) in enumerate(blur_frames.head(n_blur_show).iterrows()):
            fpath = FRAMES_OUTPUT_DIR / row["filename"]
            if fpath.exists():
                img = cv2.imread(str(fpath))
                axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
                axes[i].set_title(
                    f"BLUR | {row['timestamp_hms']} | sharp={row['sharpness']:.2f}",
                    fontsize=9
                )
            axes[i].axis("off")
        plt.suptitle(f"Motion Blur Samples — {MATCH_ID} (kept for training diversity)", fontsize=12)
        plt.tight_layout()
        plt.show()

    print(f"\nTotal frames on Drive: {len(frames_on_disk)}")
    print(f"Frames directory: {FRAMES_OUTPUT_DIR}")

## 9. Quality Distribution Analysis

Visualize the distribution of quality metrics across all extracted frames.
This helps the Lead verify consistency across different team members' videos.

In [ ]:
if len(df) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # 1. Sharpness distribution
    axes[0,0].hist(df["sharpness"], bins=30, color="steelblue", edgecolor="white")
    axes[0,0].axvline(MIN_SHARPNESS, color="red", linestyle="--", label=f"Threshold={MIN_SHARPNESS}")
    axes[0,0].set_title("Sharpness Distribution")
    axes[0,0].set_xlabel("Sharpness Score")
    axes[0,0].legend()

    # 2. Score distribution
    axes[0,1].hist(df["score"], bins=30, color="forestgreen", edgecolor="white")
    axes[0,1].set_title("Overall Quality Score")
    axes[0,1].set_xlabel("Score")

    # 3. Players per frame
    axes[0,2].hist(df["n_persons"], bins=range(0, df["n_persons"].max()+2),
                   color="coral", edgecolor="white", align="left")
    axes[0,2].set_title("Players per Frame")
    axes[0,2].set_xlabel("Number of Detected Persons")

    # 4. Max person area ratio
    axes[1,0].hist(df["max_person_area_ratio"], bins=30, color="mediumpurple", edgecolor="white")
    axes[1,0].axvline(MIN_MAX_PERSON_AREA_RATIO, color="red", linestyle="--", label=f"Min={MIN_MAX_PERSON_AREA_RATIO}")
    axes[1,0].set_title("Largest Person Size (% of frame)")
    axes[1,0].set_xlabel("Area Ratio")
    axes[1,0].legend()

    # 5. Temporal distribution
    axes[1,1].hist(df["timestamp_sec"]/60, bins=30, color="goldenrod", edgecolor="white")
    axes[1,1].set_title("Temporal Distribution")
    axes[1,1].set_xlabel("Time (minutes)")
    axes[1,1].set_ylabel("Frames")

    # 6. Pitch green ratio
    axes[1,2].hist(df["pitch_green_ratio"], bins=30, color="green", edgecolor="white")
    axes[1,2].axvline(MIN_PITCH_GREEN_RATIO, color="red", linestyle="--", label=f"Min={MIN_PITCH_GREEN_RATIO}")
    axes[1,2].set_title("Pitch Green Ratio")
    axes[1,2].set_xlabel("Green Pixel Ratio")
    axes[1,2].legend()

    plt.suptitle(f"Quality Metrics — {MATCH_ID} ({len(df)} frames)", fontsize=14)
    plt.tight_layout()
    plt.show()

    # Summary table
    print(f"\n{'='*50}")
    print(f"  QUALITY SUMMARY — {MATCH_ID}")
    print(f"{'='*50}")
    print(df[["sharpness", "score", "n_persons", "max_person_area_ratio", "pitch_green_ratio"]].describe().round(3).to_string())
else:
    print("No data to analyze.")

## 10. Upload to Roboflow (Optional)

Upload extracted frames to Roboflow for annotation.
Only run this if has confirmed the frames are ready.

In [ ]:
# --- Roboflow Upload ---
# ONLY run this cell after Lead approves your extracted frames

import getpass

ROBOFLOW_API_KEY = getpass.getpass("Paste Roboflow API Key: ")
ROBOFLOW_PROJECT = "kit-sponsor-logos"

from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
workspace = rf.workspace()

try:
    project = workspace.project(ROBOFLOW_PROJECT)
    print(f"Connected to project: {project.name}")
except Exception:
    print(f"Project '{ROBOFLOW_PROJECT}' not found. Should create it first.")
    raise

frames_to_upload = sorted(FRAMES_OUTPUT_DIR.glob(f"{MATCH_ID}_*.jpg"))
print(f"Uploading {len(frames_to_upload)} frames from {MATCH_ID}...")

# Build tag string: "M01_white_1080p,hoa"
tag_str = f"{MATCH_ID},{MEMBER_NAME}"

success, errors = 0, 0
for fp in tqdm(frames_to_upload, desc="Uploading"):
    try:
        project.upload(image_path=str(fp), split="train", tag_names=[tag_str])
        success += 1
    except Exception as e:
        errors += 1
        if errors <= 3:
            print(f"  Error: {e}")

print(f"\nUpload complete: {success} success, {errors} errors")
print(f"Annotate at: https://app.roboflow.com/{workspace.url}/{ROBOFLOW_PROJECT}/annotate")

## 11. Checklist Before Submitting

Before notifying Lead that your extraction is complete, verify:

- [ ] `TEST_MODE = False` and full video was processed
- [ ] Frames are saved to `Bradford_Bulls/frames/{MATCH_ID}/`
- [ ] Metadata CSV exists at `Bradford_Bulls/metadata/{MATCH_ID}_index.csv`
- [ ] Report exists at `Bradford_Bulls/reports/{MATCH_ID}_report.txt`
- [ ] Preview shows reasonable frames (players visible, variety of timestamps)
- [ ] No errors during extraction

**Report to Lead:**
1. Your `MATCH_ID`
2. Number of frames extracted
3. Any issues or unusual observations (e.g., "many frames rejected by pitch filter" or "video quality is low")